# RoadGuard: Late Fusion Quantitative Evaluation
**Project Context**: Master's Thesis in Computer Science Engineering
**Dataset**: Thessaloniki Road Quality Dataset (nickkotarelas/road-quality-dataset)
**Objective**: Demonstrate the performance improvements of late-sensor fusion (IMU + Vision) over single-modality branches.

---

## Environment Setup
This section installs the necessary dependencies and configures the runtime environment.

In [ ]:
# Install required libraries
!pip install ultralytics kaggle scikit-learn matplotlib pandas numpy -q
print('Dependency installation completed.')

## Google Drive Integration
Mounting Google Drive for persistent storage of trained models and evaluation artifacts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/RoadGuard_Thesis_Evaluation'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Persistent storage directory initialized: {SAVE_DIR}')

## Dataset Acquisition (Thessaloniki)
Acquiring the multi-modal dataset via Kaggle API. Requires `kaggle.json` credentials.

In [ ]:
# Authentication: Upload kaggle.json from Kaggle Account Settings
from google.colab import files
uploaded = files.upload()

import os, shutil
os.makedirs('/root/.config/kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle API credentials configured.')

In [ ]:
# Download and extract the Thessaloniki dataset
!kaggle datasets download nickkotarelas/road-quality-dataset -p /content/data/thessaloniki/ --unzip
print('Dataset extraction completed.')

## Data Preprocessing and Normalization
Normalizing the IMU data structure to ensure compatibility with the evaluation pipeline.

In [ ]:
import pandas as pd
import glob

# Locate IMU CSV files within the dataset
csv_files = glob.glob('/content/data/thessaloniki/**/*.csv', recursive=True)
if not csv_files:
    raise FileNotFoundError("No CSV data found in the dataset directory.")

IMU_CSV = csv_files[0]
df = pd.read_csv(IMU_CSV)
df.columns = [c.strip().lower() for c in df.columns]

# Column mapping to align with RoadGuard internal schema
COLUMN_MAP = {
    'x_acc':   'acc_x', 'y_acc':   'acc_y', 'z_acc':   'acc_z',
    'x_gyro':  'gyro_x', 'y_gyro':  'gyro_y', 'z_gyro':  'gyro_z',
    'class':   'label', 'target':  'label', 'anomaly': 'label'
}
df.rename(columns=COLUMN_MAP, inplace=True)

NORMALIZED_CSV = '/content/data/thessaloniki/imu_normalized.csv'
df.to_csv(NORMALIZED_CSV, index=False)
print(f'Data normalization complete. Output: {NORMALIZED_CSV}')

## Evaluation Scripts Deployment
Deploying the Python evaluation scripts designed for the RoadGuard architecture.

In [ ]:
import os
os.makedirs('/content/RoadGuard/evaluation/results', exist_ok=True)

from google.colab import files
print('Action Required: Upload the following scripts from the /evaluation directory:')
print('1. eval_imu_branch.py')
print('2. eval_vision_branch.py')
print('3. eval_late_fusion.py')
print('4. generate_report.py')
print('5. run_all.py')

uploaded = files.upload()
for fname in uploaded:
    import shutil
    shutil.copy(fname, f'/content/RoadGuard/evaluation/{fname}')
print('Evaluation scripts deployed successfully.')

## Vision Branch Model Training
Fine-tuning YOLOv8n on the figshare pothole dataset to initialize the visual detection modality.

In [ ]:
# Download training dataset
!wget -q 'https://figshare.com/ndownloader/articles/21214400/versions/3' -O /content/pothole_dataset.zip
!unzip -q /content/pothole_dataset.zip -d /content/data/pothole_dataset/

# Generate YOLOv8 configuration
yaml_content = """
path: /content/data/pothole_dataset
train: images/train
val:   images/val
names:
  0: pothole
"""
with open('/content/pothole_yolo.yaml', 'w') as f:
    f.write(yaml_content.strip())

from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/pothole_yolo.yaml', epochs=50, imgsz=640,
    batch=16, lr0=0.001, augment=True, mosaic=True, patience=10,
    project='/content/runs', name='roadguard_v1', exist_ok=True, device=0
)

BEST_PT = '/content/runs/roadguard_v1/weights/best.pt'
shutil.copy(BEST_PT, f'{SAVE_DIR}/best.pt')
print(f'Training completed. Optimal weights exported to: {SAVE_DIR}/best.pt')

## Multi-Modal Evaluation Pipeline
Executing the full evaluation pipeline: IMU analysis, Vision analysis, and Late-Fusion integration.

In [ ]:
import sys
sys.path.insert(0, '/content/RoadGuard/evaluation')
os.chdir('/content/RoadGuard')

from eval_imu_branch import main as run_imu
from eval_vision_branch import main as run_vision
from eval_late_fusion import main as run_fusion
from generate_report import main as run_report

# Step 1: IMU Evaluation
run_imu(csv_path=NORMALIZED_CSV)

# Step 2: Vision Evaluation
import glob
frame_dirs = glob.glob('/content/data/thessaloniki/**/frames', recursive=True)
FRAMES_DIR = frame_dirs[0] if frame_dirs else '/content/data/thessaloniki/frames'
run_vision(model_path=BEST_PT, frames_dir=FRAMES_DIR)

# Step 3: Fusion and Reporting
run_fusion()
run_report()
print('Evaluation pipeline execution completed.')

## Results and Visualization
Displaying the final comparative metrics and visualization for the experimental results section.

In [ ]:
from IPython.display import Image, display
display(Image('/content/RoadGuard/evaluation/results/comparison_chart.png'))

import pandas as pd
df_table = pd.read_csv('/content/RoadGuard/evaluation/results/comparison_table.csv')
print('\nExperimental Results Summary Table:')
print(df_table.to_string(index=False))

In [ ]:
# Export artifacts to Google Drive and local machine
shutil.copytree('/content/RoadGuard/evaluation/results', f'{SAVE_DIR}/results', dirs_exist_ok=True)
from google.colab import files
files.download('/content/RoadGuard/evaluation/results/comparison_chart.png')
files.download('/content/RoadGuard/evaluation/results/comparison_table.csv')
print('Evaluation artifacts exported.')